# Lab - Coordinator and Subagents in Code

Three tasks that split ShopAssist into a coordinator and two subagents, and show what each
one receives. One agent with every tool and every instruction grows confused and
expensive; small agents with a narrow job stay sharp.

| Task | What you learn |
| --- | --- |
| 1 | One function runs any agent: its own system prompt, its own tools, its own loop |
| 2 | Each subagent gets only its tools, and the facts it needs - passed explicitly |
| 3 | The coordinator has no data tools: delegating is a tool call |

**You do not need an API key.** This lab ships with a Claude simulator, so every cell
runs offline. The code you write is exactly the code you would write against the real
API - set `FUELIX_API_KEY` at home and the same notebook calls Claude for real.

**You do not write code from scratch.** Each cell already holds the code, with blanks
marked `...` and a comment telling you what goes in each one.

## Setup

Run this cell first, before anything else. Click it, then press **Shift+Enter**.

In [ ]:
# --- Lab setup (provided - just run it) ---
import shopassist_lab
from shopassist_lab import check

# Everything below this line is ordinary Claude API code.
from dotenv import load_dotenv
from anthropic import Anthropic
import json

load_dotenv()
client = Anthropic()

# The model name lives in one constant, so a later update changes one line.
MODEL = "claude-sonnet-5-5"

# One lamp order with two identical charges.
ORDERS = {
    "ORD-87654321": {"item": "lamp", "status": "delivered",
                     "delivered_days_ago": 45, "price": 60},
}
CHARGES = {
    "ORD-87654321": [{"amount": 60, "date": "day 1"},
                     {"amount": 60, "date": "day 1"}],
}


def lookup_order(order_id):
    return ORDERS.get(order_id, {"error": "order not found"})


def list_charges(order_id):
    return CHARGES.get(order_id, [])


def refund_charge(order_id, amount):
    return {"refunded": amount, "order_id": order_id}


# A small helper builds a tool from a name, a description and its fields.
def tool(name, description, fields):
    return {"name": name, "description": description,
            "input_schema": {"type": "object", "properties": fields,
                             "required": list(fields)}}


# Two tool sets: the order agent can only look up orders; the billing agent
# can list the charges of an order and refund one.
ORDER_TOOLS = [tool("lookup_order", "Look up an order by its ID.",
                    {"order_id": {"type": "string"}})]
BILLING_TOOLS = [tool("list_charges", "List the charges of an order.",
                      {"order_id": {"type": "string"}}),
                 tool("refund_charge", "Refund one charge of an order.",
                      {"order_id": {"type": "string"},
                       "amount": {"type": "number"}})]
FUNCTIONS = {"lookup_order": lookup_order, "list_charges": list_charges,
             "refund_charge": refund_charge}

# Every agent's task and tools, exactly as it received them.
RECEIVED = []

---

## Task 1 - One function runs any agent

In [ ]:
# ============================================================
# TASK 1 - One function runs any agent
# ============================================================
#
# WHAT TO DO
#   Finish run_agent(). It takes a name, a system prompt, a
#   tool list and a task, runs the loop you know, and returns
#   the text of the last reply. Then run the order agent with
#   it.
#
# WHY IT MATTERS
#   Every agent - the order agent, the billing agent, and the
#   coordinator too - is this one function with different
#   arguments. The task is the agent's whole conversation: it
#   starts with that one user message, and sees nothing else.
#
# WHERE TO SEE IT IN THE LECTURE
#   "Now the key piece: one function that runs any agent".
#
# HOW TO DO IT
#   Three blanks:
#
#       task     the agent's first and only user message
#       system   this agent's own system prompt
#       tools    this agent's own tools
# ============================================================

# BEGIN SOLUTION
def run_agent(name, system, tools, task):
    """One agent: its own system prompt, its own tools, its own loop."""
    RECEIVED.append({"agent": name, "task": task,     # provided: what this
                     "tools": [t["name"] for t in tools]})   # agent receives
    messages = [{"role": "user", "content": task}]
    while True:
        response = client.messages.create(
            model=MODEL, max_tokens=600, output_config={"effort": "low"},
            system=system, tools=tools, messages=messages)
        if response.stop_reason != "tool_use":
            break
        messages.append({"role": "assistant", "content": response.content})
        results = []
        for block in response.content:
            if block.type == "tool_use":
                output = FUNCTIONS[block.name](**block.input)
                print(f"  [{name}] {block.name}({block.input})")
                results.append({"type": "tool_result", "tool_use_id": block.id,
                                "content": json.dumps(output)})
        messages.append({"role": "user", "content": results})
    return next(b.text for b in response.content if b.type == "text")
# SCAFFOLD: def run_agent(name, system, tools, task):
# SCAFFOLD:     """One agent: its own system prompt, its own tools, its own loop."""
# SCAFFOLD:     RECEIVED.append({"agent": name, "task": task,     # provided: what this
# SCAFFOLD:                      "tools": [t["name"] for t in tools]})   # agent receives
# SCAFFOLD:     messages = [{"role": "user", "content": ...}]   # task - its only message
# SCAFFOLD:     while True:
# SCAFFOLD:         response = client.messages.create(
# SCAFFOLD:             model=MODEL, max_tokens=600, output_config={"effort": "low"},
# SCAFFOLD:             system=..., tools=..., messages=messages)   # system, tools
# SCAFFOLD:         if response.stop_reason != "tool_use":
# SCAFFOLD:             break
# SCAFFOLD:         messages.append({"role": "assistant", "content": response.content})
# SCAFFOLD:         results = []
# SCAFFOLD:         for block in response.content:
# SCAFFOLD:             if block.type == "tool_use":
# SCAFFOLD:                 output = FUNCTIONS[block.name](**block.input)
# SCAFFOLD:                 print(f"  [{name}] {block.name}({block.input})")
# SCAFFOLD:                 results.append({"type": "tool_result", "tool_use_id": block.id,
# SCAFFOLD:                                 "content": json.dumps(output)})
# SCAFFOLD:         messages.append({"role": "user", "content": results})
# SCAFFOLD:     return next(b.text for b in response.content if b.type == "text")
# END SOLUTION: replace each ... below with the value named beside it

RECEIVED.clear()
report = run_agent("order", "You check orders. Report the item, status "
                   "and delivery age in one sentence.", ORDER_TOOLS,
                   "Check order ORD-87654321.")
print("report:", report)

check("one_loop", report=report, received=RECEIVED)

---

## Task 2 - Pass the facts explicitly

In [ ]:
# ============================================================
# TASK 2 - Pass the facts explicitly
# ============================================================
#
# WHAT TO DO
#   Write the two subagents on top of run_agent(). The order
#   agent gets the order tools and a one-line job. The billing
#   agent gets the billing tools - and the facts the coordinator
#   already knows, inside its task.
#
# WHY IT MATTERS
#   It is tempting to think the subagents share the
#   coordinator's conversation. They do not. Each one starts
#   empty, with only the task text it was given. What the
#   coordinator forgets to pass, the subagent never knows. And
#   each gets only the tools for its job: the order agent
#   cannot refund anything.
#
# WHERE TO SEE IT IN THE LECTURE
#   "The order agent gets the order tools".
#
# HOW TO DO IT
#   Three blanks:
#
#       ORDER_TOOLS     the order agent's only tool
#       BILLING_TOOLS   list and refund charges
#       facts           what the coordinator already knows
# ============================================================

# BEGIN SOLUTION
def order_agent(order_id):
    return run_agent("order", "You check orders. Report the item, status "
                     "and delivery age in one sentence.", ORDER_TOOLS,
                     f"Check order {order_id}.")


def billing_agent(order_id, facts):
    return run_agent("billing", "You fix billing problems. Refund only a "
                     "duplicate charge. Report what you did in one sentence.",
                     BILLING_TOOLS, f"Order facts: {facts}\nCheck the charges "
                     f"of {order_id} and fix any duplicate.")
# SCAFFOLD: def order_agent(order_id):
# SCAFFOLD:     return run_agent("order", "You check orders. Report the item, status "
# SCAFFOLD:                      "and delivery age in one sentence.", ...,   # ORDER_TOOLS
# SCAFFOLD:                      f"Check order {order_id}.")
# SCAFFOLD:
# SCAFFOLD:
# SCAFFOLD: def billing_agent(order_id, facts):
# SCAFFOLD:     return run_agent("billing", "You fix billing problems. Refund only a "
# SCAFFOLD:                      "duplicate charge. Report what you did in one sentence.",
# SCAFFOLD:                      ...,   # BILLING_TOOLS
# SCAFFOLD:                      f"Order facts: {...}\nCheck the charges "   # facts - inside the braces
# SCAFFOLD:                      f"of {order_id} and fix any duplicate.")
# END SOLUTION: replace each ... below with the value named beside it

RECEIVED.clear()
try:
    facts = order_agent("ORD-87654321")
    print("order report:", facts)
    print("billing report:", billing_agent("ORD-87654321", facts))
except (TypeError, KeyError) as exc:   # provided: a blank left above, said plainly
    if "llipsis" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise

print()
for entry in RECEIVED:   # provided: what each agent really received
    print(entry["agent"], "got", entry["tools"], "and the task:", repr(entry["task"]))

check("explicit_context", received=RECEIVED, facts=facts)

---

## Task 3 - The coordinator delegates through agent tools

In [ ]:
# ============================================================
# TASK 3 - The coordinator delegates through agent tools
# ============================================================
#
# WHAT TO DO
#   Give the coordinator the two subagents as tools, map those
#   tool names to the agent functions, and run the customer's
#   message through the coordinator.
#
# WHY IT MATTERS
#   The coordinator talks to the customer, has no data tools at
#   all, and delegates. Delegating is a tool call: order_agent
#   with an order ID, billing_agent with an order ID and the
#   facts. Watch the order of calls - the order agent reports
#   the lamp, the billing agent gets those facts, refunds the
#   duplicate, and the coordinator writes the answer from the
#   two reports.
#
# WHERE TO SEE IT IN THE LECTURE
#   "The coordinator sees the two subagents as tools".
#
# HOW TO DO IT
#   Three blanks:
#
#       order_agent         the function behind the order_agent tool
#       billing_agent       the function behind the billing_agent tool
#       COORDINATOR_TOOLS   the coordinator's only tools
# ============================================================

COORDINATOR_TOOLS = [
    tool("order_agent", "Delegate: check an order.",
         {"order_id": {"type": "string"}}),
    tool("billing_agent", "Delegate: check and fix the charges of an "
         "order. Pass the order facts you already have.",
         {"order_id": {"type": "string"}, "facts": {"type": "string"}}),
]

RECEIVED.clear()
try:
    # BEGIN SOLUTION
    FUNCTIONS.update({"order_agent": order_agent, "billing_agent": billing_agent})

    answer = run_agent(
        "coordinator",
        "You are ShopAssist's coordinator. You have no data tools. Delegate "
        "to the order agent and the billing agent, pass each one what it "
        "needs, and answer the customer in two sentences.",
        COORDINATOR_TOOLS,
        "I was charged twice for order ORD-87654321. Please fix it.")
    # SCAFFOLD: FUNCTIONS.update({"order_agent": ...,      # order_agent - the function
    # SCAFFOLD:                   "billing_agent": ...})   # billing_agent
    # SCAFFOLD:
    # SCAFFOLD: answer = run_agent(
    # SCAFFOLD:     "coordinator",
    # SCAFFOLD:     "You are ShopAssist's coordinator. You have no data tools. Delegate "
    # SCAFFOLD:     "to the order agent and the billing agent, pass each one what it "
    # SCAFFOLD:     "needs, and answer the customer in two sentences.",
    # SCAFFOLD:     ...,   # COORDINATOR_TOOLS - the two agents, no data tools
    # SCAFFOLD:     "I was charged twice for order ORD-87654321. Please fix it.")
    # END SOLUTION: replace each ... below with the value named beside it
except (TypeError, KeyError) as exc:   # provided: a blank left above, said plainly
    if "llipsis" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise
print("ShopAssist:", answer)

check("delegate", answer=answer, received=RECEIVED)

---

## Done

Give each subagent a narrow job and only the tools for it, pass the context it needs
explicitly in the task, and let the coordinator own the customer. One note for the exam:
the exam guide calls the delegation tool **Task**; current releases of the Agent SDK call
it **Agent**. Use the exam's name on the exam.